# Zomato Restaurant Rating Predictor
### Model Training & Evaluation

This notebook covers:
- Baseline model comparison (Linear Regression, Random Forest, Gradient Boosting)
- Hyperparameter tuning with GridSearchCV
- Final model evaluation and saving

## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv('zomato_cleaned.csv')

with open('feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

X = df[feature_columns]
y = df['rate']

print(f'Features: {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target range: {y.min():.2f} - {y.max():.2f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

## 2. Baseline Model Comparison

We test three models before tuning to establish a performance baseline:
- **Linear Regression** — simple, interpretable, fast
- **Random Forest** — ensemble of decision trees, handles non-linearity
- **Gradient Boosting** — sequential ensemble, often strongest out-of-the-box

In [ ]:
def evaluate(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f'{name}')
    print(f'  R²: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}\n')
    return model

In [ ]:
lr = evaluate('Linear Regression',
              LinearRegression(),
              X_train, X_test, y_train, y_test)

rf = evaluate('Random Forest (n=100)',
              RandomForestRegressor(n_estimators=100, random_state=42),
              X_train, X_test, y_train, y_test)

gb = evaluate('Gradient Boosting (n=100)',
              GradientBoostingRegressor(n_estimators=100, random_state=42),
              X_train, X_test, y_train, y_test)

## 3. Hyperparameter Tuning — Random Forest

Random Forest showed the best baseline R². We tune it with GridSearchCV (5-fold cross validation) over:
- `n_estimators` — number of trees
- `max_depth` — how deep each tree can grow
- `min_samples_split` — minimum samples required to split a node (controls overfitting)

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring='r2'
)
grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV R²:  {grid_search.best_score_:.4f}')

## 4. Final Model Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('Tuned Random Forest — Test Set Results')
print(f'  R²:   {r2:.4f}')
print(f'  MAE:  {mae:.4f}')
print(f'  RMSE: {rmse:.4f}')

## 5. Save Model

In [ ]:
pickle.dump(best_model, open('model.pkl', 'wb'))
print('model.pkl saved successfully')